# Projeto Fictus | Análise Logística — Bloco 5: Cenários de Decisão e Recomendação

---

## Pergunta Central do Bloco
> **Fazer, não fazer ou fazer em partes — e qual é o custo de errar em cada direção?**

---

## Contexto do Bloco

Este bloco é o fechamento da tese logística. Ele consolida os custos, ganhos de SLA e riscos em cenários comparativos. O objetivo é oferecer ao tomador de decisão uma visão clara do 'Custo de Oportunidade': o que ganhamos ao internalizar vs. o que perdemos ao manter o status quo.

A recomendação final pondera a economia projetada contra a complexidade assumida, entregando um veredito baseado em margem de segurança e retorno sobre o investimento (ROI) logístico."

**Este bloco investiga:**
1. Em quais condições específicas cada cenário é superior?
2. Análise de sensibilidade — quais premissas mais movem a decisão?
3. Qual o custo de errar em cada direção?

---


## Configuração

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.ticker as mticker, matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path
try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(_base)
DIR_LOG  = BASE_DIR / "data" / "logistics"
DIR_EXPORTS = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")
COR_FRETE="#C0392B"; COR_RECEITA="#1B4F72"; COR_MARGEM="#27AE60"
COR_ALERTA="#E74C3C"; COR_NEUTRO="#7F8C8D"; COR_DESTAQUE="#E67E22"
COR_ROXO="#8E44AD"
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({"figure.dpi":150,"savefig.dpi":150,"savefig.bbox":"tight",
    "font.family":"sans-serif","axes.spines.top":False,"axes.spines.right":False})
def fmt_pct(x,pos=None): return f"{x:.1f}%"
def salvar(fig,nome):
    caminho=DIR_EXPORTS/f"{nome}.png"; fig.savefig(caminho); print(f"  -> Salvo: {caminho.name}")
print("Ambiente configurado.")


## Carregamento e Métricas Base

In [ ]:
def ler(f,**kw):
    df=pd.read_csv(DIR_LOG/f,low_memory=False,**kw); df.columns=df.columns.str.strip(); return df
log_fato=ler("log_fato.csv"); log_mensal=ler("log_mensal.csv")
log_trim=ler("log_trimestral.csv"); log_rota=ler("log_rota.csv")
for col in ["preco","valor_frete","lead_time_dias","entregue_no_prazo"]:
    if col in log_fato.columns: log_fato[col]=pd.to_numeric(log_fato[col],errors="coerce")

# Premissas centrais do modelo (todas auditaveis)
CUSTO_FIXO_MENSAL  = 180_000
CUSTO_VAR_POR_PED  = 12.50
REDUCAO_FRETE_CLI  = 0.20
AUMENTO_VOLUME     = 0.10
MELHORIA_SLA_PP    = 8
CUSTO_IMPLANTACAO  = 500_000
CUSTO_REVERSAO_EST = 300_000
MESES_MATURACAO    = 6

# Metricas base calculadas dos dados
frete_medio   = log_fato["valor_frete"].mean()
ticket_medio  = log_fato["preco"].mean()
sla_atual     = log_fato["entregue_no_prazo"].mean() * 100
n_ped_mensal  = log_mensal["n_pedidos"].mean()
pct_frete     = log_mensal["pct_frete_receita"].mean()
receita_mensal= log_mensal["receita_total"].mean()

periodos_ord = sorted(log_fato["periodo"].dropna().unique())
print(f"Base: {len(log_fato):,} pedidos | {periodos_ord[0]} a {periodos_ord[-1]}")
print(f"Frete medio: R$ {frete_medio:.2f} | SLA: {sla_atual:.1f}% | Volume medio: {n_ped_mensal:,.0f}/mes")


---

## Análise 1 — Em quais condições específicas cada cenário é superior?

> *"Não existe uma resposta certa universal para make vs buy — existe a resposta certa para as condições específicas da empresa-alvo. O Planejamento por Cenários define as fronteiras numéricas de cada decisão: acima de qual volume, abaixo de qual % de frete ao cliente e com qual SLA mínimo cada cenário passa a ser superior."*

**Framework:** Planejamento por Cenários  
**Entrega:** Matriz de condições de superioridade por cenário com fronteiras numéricas explícitas

**Como este script responde à pergunta:**
> O script calcula o benefício líquido de cada cenário em função de três variáveis simultâneas: volume mensal, % de frete ao cliente e SLA atual. Para cada combinação de valores, determina qual cenário produz o maior benefício líquido. O resultado é uma matriz de decisão tridimensional simplificada.
>
> 1. **Benefício líquido por cenário:** Barras comparando o ganho projetado de 12 meses em cada cenário, com o custo de entrada já descontado.
> 2. **Fronteiras de decisão por volume:** Curva mostrando em qual volume de pedidos cada cenário passa a ser superior — a linha que separa "manter terceirizado" de "híbrido" e "internalizar total".

**Análise do Resultado:**
Esta análise apresenta a "faixa de segurança" do investimento. Projetamos o resultado financeiro em três realidades distintas para garantir que a decisão não dependa de um cenário perfeito. Se o projeto se mantém viável mesmo no cenário conservador, o comprador tem a segurança de que o ativo possui resiliência contra oscilações de mercado.

In [ ]:
# Calcula beneficio liquido de 12 meses por cenario
ganho_frete_mensal = frete_medio * REDUCAO_FRETE_CLI * n_ped_mensal
ganho_volume_mensal= n_ped_mensal * AUMENTO_VOLUME * ticket_medio * 0.05
ganho_sla_mensal   = n_ped_mensal * (MELHORIA_SLA_PP/100) * 0.10 * 10 * ticket_medio
ganho_total_mensal = ganho_frete_mensal + ganho_volume_mensal + ganho_sla_mensal
custo_prop_mensal  = CUSTO_FIXO_MENSAL + CUSTO_VAR_POR_PED * n_ped_mensal
custo_terc_mensal  = frete_medio * n_ped_mensal
ganho_liq_mensal   = ganho_total_mensal - (custo_prop_mensal - custo_terc_mensal)

# 12 meses com maturacao
meses = np.arange(1,13)
ramp  = np.minimum(1.0, meses/MESES_MATURACAO)
ganho_12m_total   = np.sum(ganho_liq_mensal * ramp) - CUSTO_IMPLANTACAO
ganho_12m_hibrido = np.sum(ganho_liq_mensal * 0.6 * ramp) - CUSTO_IMPLANTACAO * 0.5
ganho_12m_manter  = -np.sum(frete_medio * n_ped_mensal * 0.02 * meses/12)  # custo de inacao crescente

# Fronteiras por volume
volumes = np.arange(500, n_ped_mensal * 3, 100)
be_total  = CUSTO_FIXO_MENSAL / max(0.01, frete_medio - CUSTO_VAR_POR_PED)
be_hibrido= (CUSTO_FIXO_MENSAL * 0.5) / max(0.01, frete_medio * 0.6 - CUSTO_VAR_POR_PED * 0.7)

fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle("Analise 1 - Condicoes de Superioridade por Cenario", fontsize=13, fontweight="bold")

cenarios_n = ["Manter\nterceirizado", "Modelo\nhibrido", "Internalizar\ntotal"]
ganhos     = [ganho_12m_manter, ganho_12m_hibrido, ganho_12m_total]
cores_c    = [COR_NEUTRO, COR_DESTAQUE, COR_RECEITA]
bars = axes[0].bar(cenarios_n, [g/1000 for g in ganhos], color=cores_c, alpha=0.85)
axes[0].axhline(0, color="black", linewidth=0.8)
for bar, val in zip(bars, ganhos):
    axes[0].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+(5 if val>=0 else -15),
                 f"R$ {val/1000:,.0f}K", ha="center", fontsize=9, fontweight="bold")
axes[0].set_ylabel("Beneficio liquido 12 meses (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[0].set_title(f"Beneficio Liquido em 12 Meses por Cenario\n(volume atual: {n_ped_mensal:,.0f} ped/mes)", fontsize=11)

# Fronteiras
custo_terc_v  = frete_medio * volumes
custo_prop_v  = CUSTO_FIXO_MENSAL + CUSTO_VAR_POR_PED * volumes
custo_hibr_v  = CUSTO_FIXO_MENSAL*0.5 + CUSTO_VAR_POR_PED*0.7 * volumes
axes[1].plot(volumes, custo_terc_v/1000, color=COR_NEUTRO,   linewidth=2, label="Terceirizado")
axes[1].plot(volumes, custo_prop_v/1000, color=COR_RECEITA,  linewidth=2, label="Internalizado total")
axes[1].plot(volumes, custo_hibr_v/1000, color=COR_DESTAQUE, linewidth=2, linestyle="--", label="Hibrido")
axes[1].axvline(n_ped_mensal, color=COR_MARGEM, linewidth=2, linestyle=":", label=f"Volume atual: {n_ped_mensal:,.0f}")
if 0 < be_total < volumes[-1]: axes[1].axvline(be_total, color=COR_FRETE, linewidth=1, linestyle=":", alpha=0.6, label=f"BE total: {be_total:,.0f}")
if 0 < be_hibrido < volumes[-1]: axes[1].axvline(be_hibrido, color=COR_DESTAQUE, linewidth=1, linestyle=":", alpha=0.6, label=f"BE hibrido: {be_hibrido:,.0f}")
axes[1].set_xlabel("Volume mensal de pedidos")
axes[1].set_ylabel("Custo total mensal (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[1].set_title("Fronteiras de Decisao por Volume", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
salvar(fig, "17_condicoes_superioridade_cenarios")
plt.show()

print(f"Beneficio liquido 12m — Manter    : R$ {ganho_12m_manter:,.0f}")
print(f"Beneficio liquido 12m — Hibrido   : R$ {ganho_12m_hibrido:,.0f}")
print(f"Beneficio liquido 12m — Total     : R$ {ganho_12m_total:,.0f}")
print(f"Break-even hibrido    : {be_hibrido:,.0f} ped/mes | Volume atual: {n_ped_mensal:,.0f}")
print(f"Break-even total      : {be_total:,.0f} ped/mes")


---

## Análise 2 — Análise de sensibilidade — quais premissas mais movem a decisão?

> *"As premissas do modelo têm graus diferentes de incerteza. A análise de sensibilidade identifica quais variáveis — custo fixo, custo variável, aumento de volume, redução de frete — mais impactam o resultado. Isso permite ao board concentrar esforços de validação nas premissas críticas antes de comprometer capital."*

**Framework:** Análise de Sensibilidade  
**Entrega:** Tornado chart das premissas principais — quais variáveis mais movem a decisão

**Como este script responde à pergunta:**
> O script varia cada premissa individualmente ±30% enquanto mantém as demais fixas, e mede o impacto no benefício líquido de 12 meses do cenário de internalização total. O resultado é um tornado chart — as premissas com maior impacto ficam no topo, revelando onde a incerteza é mais crítica.
>
> 1. **Tornado chart:** Cada barra representa uma premissa. O comprimento da barra é o intervalo de variação do benefício líquido quando aquela premissa vai do valor pessimista ao otimista. Barras mais longas = premissas mais críticas = onde validar antes de decidir.

**Análise do Resultado:**
Identificamos aqui os "pontos críticos de controle". Este gráfico revela qual variável externa tem o maior impacto no lucro final. Saber, por exemplo, se o negócio é mais sensível ao preço do combustível ou ao volume de vendas permite que a gestão foque seus esforços de mitigação de risco exatamente onde a exposição é maior.

In [ ]:
# Premissas base e variacao de +-30%
base_ganho = ganho_12m_total

def calc_ganho(cf=CUSTO_FIXO_MENSAL, cv=CUSTO_VAR_POR_PED,
               rf=REDUCAO_FRETE_CLI, av=AUMENTO_VOLUME, ss=MELHORIA_SLA_PP):
    gf = frete_medio * rf * n_ped_mensal
    gv = n_ped_mensal * av * ticket_medio * 0.05
    gs = n_ped_mensal * (ss/100) * 0.10 * 10 * ticket_medio
    cp = cf + cv * n_ped_mensal
    ct = frete_medio * n_ped_mensal
    gl = gf + gv + gs - (cp - ct)
    ramp = np.minimum(1.0, np.arange(1,13)/MESES_MATURACAO)
    return np.sum(gl * ramp) - CUSTO_IMPLANTACAO

premissas = [
    ("Custo fixo mensal",   lambda v: calc_ganho(cf=v), CUSTO_FIXO_MENSAL),
    ("Custo variavel/ped",  lambda v: calc_ganho(cv=v), CUSTO_VAR_POR_PED),
    ("Reducao frete client",lambda v: calc_ganho(rf=v), REDUCAO_FRETE_CLI),
    ("Aumento de volume",   lambda v: calc_ganho(av=v), AUMENTO_VOLUME),
    ("Melhoria SLA (pp)",   lambda v: calc_ganho(ss=v), MELHORIA_SLA_PP),
]
VAR = 0.30  # variacao de +-30%
resultados = []
for nome, func, base in premissas:
    g_pess = func(base * (1 + VAR)) if "fixo" in nome.lower() or "variavel" in nome.lower() else func(base * (1 - VAR))
    g_otim = func(base * (1 - VAR)) if "fixo" in nome.lower() or "variavel" in nome.lower() else func(base * (1 + VAR))
    resultados.append({"premissa": nome, "pessimista": g_pess, "otimista": g_otim,
                       "amplitude": abs(g_otim - g_pess)})

df_tornado = pd.DataFrame(resultados).sort_values("amplitude")

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title("Tornado Chart — Sensibilidade das Premissas ao Beneficio Liquido de 12 Meses",
             fontsize=12, fontweight="bold")

y = range(len(df_tornado))
for i, (_, row) in enumerate(df_tornado.iterrows()):
    lo = min(row["pessimista"], row["otimista"])
    hi = max(row["pessimista"], row["otimista"])
    ax.barh(i, hi - lo, left=lo - base_ganho, height=0.6,
            color=COR_RECEITA if hi - base_ganho > 0 else COR_FRETE, alpha=0.8)
ax.axvline(0, color="black", linewidth=1.5, linestyle="--", label=f"Base: R$ {base_ganho/1000:,.0f}K")
ax.set_yticks(y)
ax.set_yticklabels(df_tornado["premissa"], fontsize=10)
ax.set_xlabel("Variacao no Beneficio Liquido 12m em relacao a base (R$)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v/1000:,.0f}K"))
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
salvar(fig, "18_tornado_chart_sensibilidade")
plt.show()

print("Ranking de premissas por impacto:")
for _,r in df_tornado.sort_values("amplitude", ascending=False).iterrows():
    print(f"  {r['premissa']:<30} amplitude: R$ {r['amplitude']:,.0f} | pess: R$ {r['pessimista']:,.0f} | otim: R$ {r['otimista']:,.0f}")


---

## Análise 3 — Qual o custo de errar em cada direção?

> *"A decisão de make vs buy é assimétrica: os custos de errar em cada direção são diferentes. Internalizar cedo demais imobiliza capital antes de ter volume suficiente. Postergar demais mantém o frete alto ao cliente por mais tempo, freando conversão e crescimento. A comparação dos dois custos por trimestre revela ao board a assimetria real da decisão e o momento ótimo de agir."*

**Framework:** Custo de Oportunidade + Análise de Sensibilidade  
**Entrega:** Comparação do custo de antecipação versus custo de postergação por trimestre

**Como este script responde à pergunta:**
> O script calcula, trimestre a trimestre, quanto custa antecipar a decisão (investir antes de ter volume suficiente) e quanto custa postergar (deixar o frete alto ao cliente por mais tempo). O ponto onde as duas curvas se cruzam é o momento ótimo de decisão — antes disso, esperar custa menos; depois disso, agir custa menos.
>
> 1. **Custo de antecipação por trimestre:** Quanto o negócio perde se internalizar N trimestres antes do momento ótimo.
> 2. **Custo de postergação por trimestre:** Quanto o negócio perde se postergar N trimestres depois do momento ótimo.

**Análise do Resultado:**
Esta análise revela o "ponto de equilíbrio temporal" da decisão. Identificamos que o custo de agir cedo demais (imobilização de capital sem volume) e o custo de demorar a agir (perda de competitividade por frete alto) não são iguais. O gráfico mostra o momento exato onde o custo de manter o modelo atual supera o custo de investir na nova estrutura. Para o comprador, esta visão é o guia definitivo de timing: ela prova matematicamente se a janela de oportunidade para a internalização está aberta agora ou se o cenário recomenda aguardar alguns trimestres para minimizar o prejuízo do erro.

In [ ]:
# Custo de postergacao: cada trimestre adicional de frete alto ao cliente
custo_post_trim = receita_mensal * 0.015 * 3  # ~1.5% de perda de conversao por trimestre

# Custo de antecipacao: imobilizar capital antes de ter volume para o break-even
vol_break_even = CUSTO_FIXO_MENSAL / max(0.01, frete_medio - CUSTO_VAR_POR_PED)
gap_volume = max(0, vol_break_even - n_ped_mensal)
custo_ant_trim = gap_volume * (CUSTO_FIXO_MENSAL / n_ped_mensal) * 3  # custo do fixo ocioso por trimestre

trimestres = np.arange(1, 9)
custo_post = custo_post_trim * trimestres
custo_ant  = custo_ant_trim * np.maximum(0, trimestres - 2)  # primeiros 2 tri ainda fazem sentido

# Trimestre otimo: onde custo acumulado de postergar supera custo de antecipar
custo_net_post  = np.cumsum(custo_post_trim * np.ones(8))
custo_net_ant   = custo_ant_trim * np.ones(8)
trim_otimo = next((i+1 for i,(p,a) in enumerate(zip(custo_net_post, custo_net_ant)) if p > a), 8)

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title("Analise 3 - Custo de Errar: Antecipar vs Postergar por Trimestre", fontsize=12, fontweight="bold")
ax.plot(trimestres, custo_net_post/1000, color=COR_FRETE,   linewidth=2, marker="o", markersize=5, label="Custo acumulado de postergar")
ax.plot(trimestres, custo_net_ant/1000,  color=COR_RECEITA, linewidth=2, marker="s", markersize=5, label="Custo de antecipar (por trim.)")
ax.axvline(trim_otimo, color=COR_MARGEM, linewidth=2, linestyle=":", label=f"Momento otimo: trimestre {trim_otimo}")
ax.fill_between(trimestres, custo_net_post/1000, custo_net_ant/1000,
                where=custo_net_post > custo_net_ant, alpha=0.15, color=COR_FRETE, label="Zona de postergacao custosa")
ax.set_xlabel("Trimestres a partir de agora")
ax.set_ylabel("Custo acumulado (R$ mil)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
salvar(fig, "19_custo_errar_antecipar_postergar")
plt.show()

print(f"Custo de postergar por trimestre : R$ {custo_post_trim:,.0f}")
print(f"Custo de antecipar por trimestre : R$ {custo_ant_trim:,.0f}")
print(f"Momento otimo de decisao         : trimestre {trim_otimo}")
print(f"Assimetria: {'postergar custa mais' if custo_post_trim > custo_ant_trim else 'antecipar custa mais'}")


---
## Síntese e Recomendação Final

> **Limitações desta análise:** o benefício líquido de 12 meses é calculado com premissas declaradas e auditáveis — não com cotações reais de operação logística própria. O tornado chart mede sensibilidade às premissas do modelo, não incerteza empírica do mercado. O custo de postergação assume perda de conversão de 1,5% por trimestre — estimativa conservadora não validada empiricamente para este dataset. A recomendação é baseada em dados transacionais históricos e não substitui due diligence operacional, contratual ou jurídica sobre o modelo logístico vigente.


In [ ]:
# ─── Scores por dimensao ──────────────────────────────────────────────────────
# Bloco 1: modelo atual
_pct_frete   = log_mensal["pct_frete_receita"].mean()
_tend_up     = np.polyfit(range(len(log_mensal)), log_mensal["pct_frete_receita"].fillna(_pct_frete), 1)[0] > 0.1
_b1_score    = 1 if _pct_frete > 22 and _tend_up else 2 if _pct_frete > 20 or _tend_up else 3
_b1_sinal    = ("MODELO NO LIMITE" if _b1_score==1 else "MODELO SOB PRESSAO" if _b1_score==2 else "MODELO SAUDAVEL")
_b1_cor      = COR_ALERTA if _b1_score==1 else COR_DESTAQUE if _b1_score==2 else COR_MARGEM

# Bloco 2: viabilidade economica
_ganho_ok    = ganho_12m_total > 0
_payback_ok  = ganho_12m_hibrido > 0
_b2_score    = 3 if (_ganho_ok and ganho_12m_total > CUSTO_IMPLANTACAO) else 2 if _ganho_ok else 1
_b2_sinal    = ("INTERNALIZACAO VIAVEL" if _b2_score==3 else "HIBRIDO VIAVEL" if _b2_score==2 else "INTERNALIZACAO PREMATURA")
_b2_cor      = COR_MARGEM if _b2_score==3 else COR_DESTAQUE if _b2_score==2 else COR_ALERTA

# Bloco 3: SLA
_sla_ok      = sla_atual >= 85
_b3_score    = 3 if sla_atual >= 90 else 2 if sla_atual >= 80 else 1
_b3_sinal    = ("SLA NO BENCHMARK" if _b3_score==3 else "SLA ABAIXO DA META" if _b3_score==2 else "SLA CRITICO")
_b3_cor      = COR_MARGEM if _b3_score==3 else COR_DESTAQUE if _b3_score==2 else COR_ALERTA

# Bloco 4: escalabilidade
_b4_score    = 2  # moderado por default sem vol_ruptura calculado aqui
_b4_sinal    = "RISCO MODERADO — monitorar escalabilidade"
_b4_cor      = COR_DESTAQUE

# Bloco 5: cenarios (este bloco)
_melhor_cen  = max([("manter",ganho_12m_manter),("hibrido",ganho_12m_hibrido),("total",ganho_12m_total)], key=lambda x:x[1])
_b5_score    = 3 if _melhor_cen[0]!="manter" else 2
_b5_sinal    = f"CENARIO SUPERIOR: {_melhor_cen[0].upper()}"
_b5_cor      = COR_MARGEM if _b5_score==3 else COR_DESTAQUE

blocos = [
    {"bloco":"Bloco 1 — Diagnostico do Modelo", "sinal":_b1_sinal, "cor":_b1_cor, "score":_b1_score,
     "achado":f"Frete ao cliente: {_pct_frete:.1f}% da receita. Tendencia: {'crescente' if _tend_up else 'estavel'}."},
    {"bloco":"Bloco 2 — Viabilidade Economica",  "sinal":_b2_sinal, "cor":_b2_cor, "score":_b2_score,
     "achado":f"Beneficio 12m (total): R$ {ganho_12m_total:,.0f} | (hibrido): R$ {ganho_12m_hibrido:,.0f}"},
    {"bloco":"Bloco 3 — SLA e Satisfacao",        "sinal":_b3_sinal, "cor":_b3_cor, "score":_b3_score,
     "achado":f"SLA atual: {sla_atual:.1f}% | Com internalizacao: {sla_atual+MELHORIA_SLA_PP:.1f}%"},
    {"bloco":"Bloco 4 — Escalabilidade/Riscos",   "sinal":_b4_sinal, "cor":_b4_cor, "score":_b4_score,
     "achado":f"Custo reversao: R$ {CUSTO_REVERSAO_EST:,.0f} (total) | R$ {CUSTO_REVERSAO_EST*0.4:,.0f} (hibrido)"},
    {"bloco":"Bloco 5 — Cenarios",                "sinal":_b5_sinal, "cor":_b5_cor, "score":_b5_score,
     "achado":f"Cenario superior: {_melhor_cen[0]} | Ganho: R$ {_melhor_cen[1]:,.0f} em 12m"},
]

# Grafico de consolidacao
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("RELATORIO FINAL — Consolidacao dos 5 Blocos", fontsize=14, fontweight="bold")

nomes_b = [b["bloco"].split(" — ")[1] for b in blocos]
scores_b= [b["score"] for b in blocos]
cores_b = [b["cor"] for b in blocos]
bars = axes[0].barh(nomes_b, scores_b, color=cores_b, alpha=0.85)
axes[0].set_xlim(0, 3.5); axes[0].set_xticks([1,2,3])
axes[0].set_xticklabels(["Risco Alto","Atencao","Positivo"], fontsize=9)
axes[0].axvline(1.5, color=COR_NEUTRO, linewidth=0.5, alpha=0.4)
axes[0].axvline(2.5, color=COR_NEUTRO, linewidth=0.5, alpha=0.4)
for bar, b in zip(bars, blocos):
    axes[0].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                 b["sinal"][:40], va="center", fontsize=7.5, color="#333333")
axes[0].set_title("Score por Bloco Decisorio", fontsize=11)

# Comparacao financeira dos cenarios
axes[1].bar(["Manter\nterceirizado","Modelo\nhibrido","Internalizar\ntotal"],
            [ganho_12m_manter/1000, ganho_12m_hibrido/1000, ganho_12m_total/1000],
            color=[COR_NEUTRO, COR_DESTAQUE, COR_RECEITA], alpha=0.85)
axes[1].axhline(0, color="black", linewidth=0.8)
for i, val in enumerate([ganho_12m_manter, ganho_12m_hibrido, ganho_12m_total]):
    axes[1].text(i, val/1000+(5 if val>=0 else -15), f"R$ {val/1000:,.0f}K",
                 ha="center", fontsize=9, fontweight="bold")
axes[1].set_ylabel("Beneficio liquido 12 meses (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[1].set_title("Beneficio Liquido por Cenario — 12 Meses", fontsize=11)
plt.tight_layout()
salvar(fig, "20_relatorio_final_consolidado")
plt.show()

# ─── Recomendacao dinamica ──────────────────────────────────────────────────
score_medio = sum(b["score"] for b in blocos) / len(blocos)
n_score1    = sum(1 for b in blocos if b["score"]==1)
n_score3    = sum(1 for b in blocos if b["score"]==3)

if n_score1 >= 2:
    decisao = "NAO RECOMENDADA — riscos estruturais bloqueantes"
    cond_p1 = "NAO ATENDIDA — riscos superam o potencial de valor"
elif _melhor_cen[0] == "hibrido" or (n_score1 == 1 and score_medio < 2.5):
    decisao = "MODELO HIBRIDO RECOMENDADO — internalizacao total prematura no volume atual"
    cond_p1 = "PARCIALMENTE ATENDIDA — hibrido endereça a condicionante com menor risco"
elif score_medio >= 2.5:
    decisao = "INTERNALIZACAO TOTAL RECOMENDADA — condicao da aquisicao atendida"
    cond_p1 = "ATENDIDA — a internalizacao resolve a barreira de conversao identificada no Projeto 1"
else:
    decisao = "CONDICIONAL — aguardar crescimento de volume antes de internalizar totalmente"
    cond_p1 = "PARCIALMENTE ATENDIDA — modelo hibrido como passo intermediario"

print("=" * 70)
print("RECOMENDACAO FINAL — FICTUS LOGISTICS")
print("=" * 70)
print(f"\nDecisao: {decisao}")
print(f"\nCondicao do Projeto 1 (FICTUS Retail):")
print(f"  {cond_p1}")
print("\n[ SCORES POR BLOCO ]")
NIVEL = {3:"POSITIVO",2:"ATENCAO",1:"RISCO"}
for b in blocos:
    print(f"  {NIVEL[b['score']]}  {b['bloco']:<40} {b['achado'][:60]}")
print("\n[ NUMEROS DA DECISAO ]")
print(f"  Cenario superior (12m)  : {_melhor_cen[0].upper()} — R$ {_melhor_cen[1]:,.0f}")
print(f"  Momento otimo           : trimestre {trim_otimo}")
print(f"  Premissa mais critica   : {df_tornado.iloc[-1]['premissa']}")
print(f"  Custo de reversao       : R$ {CUSTO_REVERSAO_EST:,.0f} (total) | R$ {CUSTO_REVERSAO_EST*0.4:,.0f} (hibrido)")
print("=" * 70)
print("\nFICTUS | Análise Logística concluída.")
print("Proximo passo: FICTUS | Análise Financeira — Viabilidade de internalização de crédito.")


---

## Summary Log — Input para o Relatório de Recomendação

> *Execute a célula abaixo após rodar todos os blocos. O output gerado deve ser copiado e colado no Prompt Mestre do Relatório de Recomendação para gerar o parecer executivo consolidado das três partes da análise.*

---


In [ ]:
# ─── SUMMARY LOG — FICTUS | Análise Logística ────────────────────────────────
# Cole o output desta célula no Prompt Mestre do Relatório de Recomendação.
# Execute APÓS rodar todos os blocos anteriores deste notebook.

import io, sys
from pathlib import Path
from datetime import datetime

def salvar_summary_log(conteudo: str, frente: str) -> None:
    """Salva o Summary Log em summary_logs/ com timestamp."""
    pasta = BASE_DIR / 'summary_logs'
    pasta.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    nome = f'summary_log_{frente}_{timestamp}.txt'
    (pasta / nome).write_text(conteudo, encoding='utf-8')
    print(f'✅ Summary Log salvo em: summary_logs/{nome}')

# Captura o output dos prints para salvar em arquivo
_buffer = io.StringIO()
_stdout_orig = sys.stdout
sys.stdout = _buffer

try:
    # ─── SUMMARY LOG — FICTUS | Análise Logística ────────────────────────────────
    # Cole o output desta célula no Prompt Mestre do Relatório de Recomendação.
    # Execute APÓS rodar todos os blocos anteriores deste notebook.
    
    print("=" * 70)
    print("FICTUS | ANÁLISE LOGÍSTICA — SUMMARY LOG")
    print("=" * 70)
    
    try:
        _n_pedidos_log        = log_fato["id_pedido"].nunique()
        _frete_medio_log      = log_fato["valor_frete"].mean()
        _pct_frete_log        = log_mensal["pct_frete_receita"].mean()
        _lead_medio_log       = log_fato["lead_time_dias"].mean()
        _pct_no_prazo_log     = log_fato["entregue_no_prazo"].mean() * 100
        _nota_media_log       = log_fato["nota_review"].mean()
        _score_medio_log      = sum(b["score"] for b in blocos) / len(blocos)
        _decisao_log          = decisao
    
        print(f"""
    [PARTE 2 — ANÁLISE LOGÍSTICA]
    
    Período analisado         : {log_mensal["ano_mes"].min()} a {log_mensal["ano_mes"].max()}
    Total de pedidos          : {_n_pedidos_log:,}
    Frete médio por pedido    : R$ {_frete_medio_log:.2f}
    Frete como % da receita   : {_pct_frete_log:.1f}%
    Lead time médio           : {_lead_medio_log:.1f} dias
    SLA (% no prazo)          : {_pct_no_prazo_log:.1f}%
    Nota média de review      : {_nota_media_log:.2f}
    
    --- CENÁRIO RECOMENDADO ---
    Decisão desta parte       : {_decisao_log}
    Cenário superior          : {_melhor_cen[0].upper() if "_melhor_cen" in dir() else "ver síntese"}
    Premissa crítica          : {df_tornado.iloc[-1]["premissa"] if "df_tornado" in dir() else "ver análise de sensibilidade"}
    Custo de reversão         : R$ {CUSTO_REVERSAO_EST:,.0f} (total)
    
    --- SCORES POR BLOCO ---""")
    
        for b in blocos:
            nome = b["bloco"] if isinstance(b["bloco"], str) else str(b["bloco"])
            print(f"  {b['score']}/3  {nome}")
    
        print(f"""
    Score médio               : {_score_medio_log:.1f}/3
    
    --- DECLARAÇÃO DE ESCOPO ---
    Análise baseada em dados transacionais (Olist dataset).
    Transportadoras não identificadas individualmente — risco de concentração avaliado por concentração geográfica. Custos são premissas declaradas e auditáveis.
    Não inclui: jurídico, contábil, contratos de terceirização vigentes.
    """)
    
    except NameError as e:
        print(f"[AVISO] Execute todos os blocos anteriores antes de gerar o Summary Log.")
        print(f"  Variável não encontrada: {e}")
    
    print("=" * 70)
    

finally:
    sys.stdout = _stdout_orig
    _conteudo = _buffer.getvalue()
    print(_conteudo)  # imprime normalmente na tela
    salvar_summary_log(_conteudo, 'logistica')


---
*Este notebook encerra a **Fase 2 — Logística** do Projeto Fictus.*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
